# Generative AI & Large Language Models (LLMs)

A comprehensive exploration of how modern AI systems generate text, including real model inference with transformers.

📺 **Video Lecture:** [https://youtu.be/q728r276RA0](https://youtu.be/q728r276RA0)

## Setup & Imports

**Prerequisites:** `pip install transformers torch numpy matplotlib pandas`

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings('ignore')

# Check device (CPU/GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Load a Real LLM: DistilGPT-2

DistilGPT-2 is a lightweight, CPU-friendly version of GPT-2. It has ~82M parameters and is ideal for demonstration purposes.

In [ ]:
# Load the tokenizer and model
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

# Set model to evaluation mode
model.eval()

# Model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\\nModel architecture:")
print(model)

## 2. Tokenization Deep Dive

Tokenization converts raw text into token IDs that the model understands.

In [ ]:
# Example text
text = "Artificial Intelligence is transforming technology."

# Tokenize
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text, return_tensors='pt')

print(f"Text: {text}")
print(f"\\nTokens: {tokens}")
print(f"Token IDs: {token_ids.tolist()[0]}")
print(f"Number of tokens: {len(tokens)}")

# Decode back
decoded = tokenizer.decode(token_ids[0])
print(f"\\nDecoded: {decoded}")

# Vocabulary information
print(f"\\nVocabulary size: {tokenizer.vocab_size:,}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

# Show some example tokens
print(f"\\nExample token ID -> token mappings:")
for token_id in token_ids[0][:10].tolist():
    token = tokenizer.decode([token_id])
    print(f"  {token_id:4d} -> '{token}'")

## 3. Text Generation with Different Strategies

Demonstrate greedy decoding vs. sampling-based generation.

In [ ]:
prompt = "The future of artificial intelligence"

# Encode prompt
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# Greedy decoding
print("=== GREEDY DECODING ===")
with torch.no_grad():
    output_greedy = model.generate(
        input_ids,
        max_length=50,
        num_beams=1,
        do_sample=False,
        temperature=1.0
    )
generated_greedy = tokenizer.decode(output_greedy[0], skip_special_tokens=True)
print(generated_greedy)

# Sampling with temperature
print("\\n=== SAMPLING (temperature=0.7) ===")
with torch.no_grad():
    output_sample = model.generate(
        input_ids,
        max_length=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.95
    )
generated_sample = tokenizer.decode(output_sample[0], skip_special_tokens=True)
print(generated_sample)

# Beam search
print("\\n=== BEAM SEARCH (num_beams=5) ===")
with torch.no_grad():
    output_beam = model.generate(
        input_ids,
        max_length=50,
        num_beams=5,
        no_repeat_ngram_size=2
    )
generated_beam = tokenizer.decode(output_beam[0], skip_special_tokens=True)
print(generated_beam)

## 4. Temperature Effect on Generation

Temperature controls randomness: lower = more deterministic, higher = more creative.

In [ ]:
prompt = "Machine learning models learn from"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

temperatures = [0.3, 0.7, 1.0, 1.5]
results = {}

print(f"Prompt: {prompt}\\n")

for temp in temperatures:
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=40,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
            num_return_sequences=1
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    results[temp] = generated
    print(f"Temperature {temp} (Deterministic -> Creative):")
    print(f"  {generated}\\n")

## 5. Top-K and Top-P Sampling

Top-K restricts choices to the K most likely tokens. Top-P (nucleus sampling) restricts to a cumulative probability threshold.

In [ ]:
prompt = "Deep learning has revolutionized"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

print(f"Prompt: {prompt}\\n")

# Top-K Sampling
print("=== TOP-K SAMPLING ===")
for k in [10, 50, 100]:
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=40,
            do_sample=True,
            top_k=k,
            top_p=1.0
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Top-K={k}: {generated}")

# Top-P Sampling
print("\\n=== TOP-P SAMPLING ===")
for p in [0.6, 0.85, 0.95]:
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=40,
            do_sample=True,
            top_k=50,
            top_p=p
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Top-P={p}: {generated}")

## 6. Perplexity Calculation

Perplexity measures how well a model predicts a text. Lower perplexity = better predictions.

In [ ]:
def calculate_perplexity(model, tokenizer, text):
    """Calculate perplexity of a given text."""
    input_ids = tokenizer.encode(text, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss
    
    perplexity = torch.exp(loss).item()
    return perplexity

# Test sentences
grammatical = "The quick brown fox jumps over the lazy dog."
grammatical2 = "Natural language processing is a subfield of artificial intelligence."
ungrammatical = "fox brown the quick over jumps dog lazy."
random_words = "elephant purple mathematics quantum coding breakfast."

sentences = {
    "Grammatical 1": grammatical,
    "Grammatical 2": grammatical2,
    "Ungrammatical": ungrammatical,
    "Random words": random_words
}

print("Perplexity Results (lower = better predictions):\\n")
perplexities = []

for label, text in sentences.items():
    ppl = calculate_perplexity(model, tokenizer, text)
    perplexities.append(ppl)
    print(f"{label:20s}: {ppl:8.2f}")
    print(f"  Text: {text}\\n")

# Visualization
plt.figure(figsize=(10, 5))
colors = ['green', 'green', 'orange', 'red']
plt.bar(sentences.keys(), perplexities, color=colors, alpha=0.7, edgecolor='black')
plt.ylabel('Perplexity', fontsize=12)
plt.title('Model Perplexity on Different Text Types', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Prompt Engineering with Real Model

Demonstrate zero-shot and few-shot prompting patterns.

In [ ]:
# Zero-shot prompting
print("=== ZERO-SHOT PROMPTING ===")
zero_shot_prompt = "Summarize what machine learning is:"
input_ids = tokenizer.encode(zero_shot_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    output = model.generate(input_ids, max_length=60, do_sample=True, temperature=0.7)
generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Prompt: {zero_shot_prompt}")
print(f"Response: {generated}\\n")

# Few-shot prompting
print("=== FEW-SHOT PROMPTING ===")
few_shot_prompt = """Question: What is machine learning?
Answer: Machine learning is a field of AI where computers learn from data.

Question: What is deep learning?
Answer:"""
input_ids = tokenizer.encode(few_shot_prompt, return_tensors='pt').to(device)
with torch.no_grad():
    output = model.generate(input_ids, max_length=80, do_sample=True, temperature=0.7)
generated = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Prompt:\\n{few_shot_prompt}")
print(f"\\nResponse:\\n{generated}")

## 8. Token Probability Distribution

Analyze the probability distribution over the next token for a given prompt.

In [ ]:
prompt = "Artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# Get model predictions for next token
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits[0, -1, :]
    probabilities = torch.softmax(logits, dim=-1)

# Get top-10 candidates
top_k = 10
top_probs, top_indices = torch.topk(probabilities, top_k)

print(f"Prompt: {prompt}")
print(f"\\nTop-10 next token candidates:\\n")

tokens_list = []
probs_list = []

for prob, idx in zip(top_probs, top_indices):
    token = tokenizer.decode([idx.item()])
    prob_val = prob.item()
    tokens_list.append(token)
    probs_list.append(prob_val)
    print(f"  {token:20s}: {prob_val:6.2%}")

# Visualization
plt.figure(figsize=(12, 5))
plt.barh(range(len(tokens_list)), probs_list, color='steelblue', edgecolor='black')
plt.yticks(range(len(tokens_list)), tokens_list)
plt.xlabel('Probability', fontsize=11)
plt.title(f'Top-10 Next Token Predictions for: \"{prompt}\"', fontsize=13, fontweight='bold')
plt.xlim([0, max(probs_list) * 1.1])
for i, v in enumerate(probs_list):
    plt.text(v + 0.002, i, f'{v:.2%}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 9. Generation Parameters Comparison

Compare outputs across different generation strategies on the same prompt.

In [ ]:
prompt = "Technology companies are increasingly using"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

strategies = {
    "Greedy": {
        "do_sample": False,
        "num_beams": 1,
        "temperature": 1.0
    },
    "Beam Search (5)": {
        "do_sample": False,
        "num_beams": 5,
        "temperature": 1.0
    },
    "Sampling (T=0.5)": {
        "do_sample": True,
        "temperature": 0.5,
        "top_p": 0.95
    },
    "Sampling (T=1.0)": {
        "do_sample": True,
        "temperature": 1.0,
        "top_p": 0.95
    },
    "Top-K (K=20)": {
        "do_sample": True,
        "top_k": 20,
        "top_p": 1.0,
        "temperature": 0.7
    },
    "Top-P (P=0.8)": {
        "do_sample": True,
        "top_k": 50,
        "top_p": 0.8,
        "temperature": 0.7
    }
}

results_df = []

print(f"Prompt: {prompt}\\n")
print("=" * 80)

for strategy_name, params in strategies.items():
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=55,
            **params
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    generated_text = generated[len(prompt):].strip()
    
    print(f"\\n{strategy_name}:")
    print(f"  {generated}")
    
    results_df.append({
        "Strategy": strategy_name,
        "Generated Text Length": len(generated),
        "Continuation Length": len(generated_text)
    })

print("\\n" + "=" * 80)

# Summary table
df_summary = pd.DataFrame(results_df)
print("\\nSummary:")
print(df_summary.to_string(index=False))

## 10. LLM Architecture Comparison

Comparison of key architectures used in modern language models.

In [ ]:
# Architecture comparison table
architectures = {
    "Model": ["GPT (GPT-2, GPT-3)", "BERT", "T5", "LLaMA", "DistilGPT-2"],
    "Encoder": ["No", "Yes", "Yes", "No", "No"],
    "Decoder": ["Yes", "No", "Yes", "Yes", "Yes"],
    "Training Objective": [
        "Causal Language Modeling",
        "Masked Language Modeling",
        "Sequence-to-Sequence",
        "Causal Language Modeling",
        "Causal Language Modeling"
    ],
    "Primary Use Case": [
        "Text generation, chat",
        "Classification, embeddings",
        "Translation, summarization",
        "Text generation, instruction following",
        "Lightweight text generation"
    ],
    "Parameters (Typical)": [
        "125M - 175B",
        "110M - 340M",
        "60M - 11B",
        "7B - 65B",
        "82M"
    ]
}

df_arch = pd.DataFrame(architectures)
print("\\nLLM Architecture Comparison:")
print(df_arch.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Encoder/Decoder architecture
arch_types = ["Encoder Only\\n(BERT)", "Decoder Only\\n(GPT, LLaMA)", "Encoder-Decoder\\n(T5)"]
arch_counts = [1, 2, 1]
colors_arch = ['#FF6B6B', '#4ECDC4', '#45B7D1']

axes[0].bar(arch_types, arch_counts, color=colors_arch, alpha=0.7, edgecolor='black', width=0.6)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].set_title('Architecture Types in Common LLMs', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 3])

# Training objectives
objectives = ["Causal LM", "Masked LM", "Seq2Seq"]
model_counts = [3, 1, 1]
colors_obj = ['#45B7D1', '#FF6B6B', '#96CEB4']

axes[1].bar(objectives, model_counts, color=colors_obj, alpha=0.7, edgecolor='black', width=0.6)
axes[1].set_ylabel('Number of Models', fontsize=11)
axes[1].set_title('Training Objectives Distribution', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 3.5])

plt.tight_layout()
plt.show()

## 11. Interview Takeaways

Key concepts and insights from this deep dive into Generative AI and LLMs.

### Core Concepts

1. **Tokenization**: Text is converted to token IDs, which the model processes. Vocabulary size matters for efficiency.

2. **Causal Language Modeling**: Next-token prediction. Models learn P(token_i | token_1, ..., token_{i-1}).

3. **Temperature**: Controls randomness in sampling.
   - Low T (< 1.0): More deterministic, focuses on high-probability tokens
   - High T (> 1.0): More creative, flattens probability distribution

4. **Decoding Strategies**:
   - **Greedy**: Always pick the highest-probability token (fast, may be repetitive)
   - **Beam Search**: Explore multiple paths, pick the best sequence (slower, better quality)
   - **Sampling**: Random draw from distribution (diverse, less deterministic)
   - **Top-K**: Restrict to K highest-probability tokens
   - **Top-P (Nucleus)**: Include tokens until cumulative probability reaches P

5. **Perplexity**: Measures how well the model predicts text. Lower = better fit to the language.

6. **Prompt Engineering**:
   - Zero-shot: Give the task directly
   - Few-shot: Show examples before the actual task
   - System prompts: Set context/role for the model

### Architecture Patterns

- **Decoder-only (GPT, LLaMA)**: Optimized for generation. Can't see future context.
- **Encoder-only (BERT)**: Bidirectional. Good for understanding/classification.
- **Encoder-Decoder (T5)**: Best for translation, summarization.

### Practical Insights

- Model size vs. capability: Larger models (generally) perform better, but require more compute.
- Scaling laws: Performance improves with model size, data size, and compute.
- Few-shot learning: LLMs can adapt to new tasks from examples.
- Hallucinations: Models generate plausible but false information.
- Context window: Limited length of input the model can process at once.

### Common Interview Questions

1. **How does temperature affect generation?** Higher temperature increases randomness; lower makes it deterministic.

2. **What's the difference between greedy and beam search?** Greedy picks the top token each step; beam search explores multiple paths.

3. **Why is perplexity important?** It quantifies how well the model predicts text—lower perplexity = better language modeling.

4. **What is prompt engineering?** Crafting input prompts to get better model outputs (zero-shot, few-shot, chain-of-thought).

5. **Why use sampling instead of greedy decoding?** Sampling produces diverse outputs; greedy is deterministic but may get stuck in loops.

6. **Explain attention mechanism.** Allows the model to focus on different parts of the input when making predictions.

7. **What are transformers?** Neural architecture using self-attention; foundation for modern LLMs.

8. **How does tokenization affect model performance?** Vocabulary size, token efficiency, and special tokens all impact downstream performance.

9. **What is the difference between training and inference?** Training: learn weights. Inference: use fixed weights to generate output.

10. **What challenges do LLMs face?** Hallucinations, context limits, computational cost, bias, data privacy.

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>